In [1]:
# ============================================================
# FULL NOTEBOOK — Run as ONE cell after fresh runtime restart
# ============================================================

# --- INSTALL ---
!pip install -q transformers datasets accelerate peft trl bitsandbytes gradio

# --- IMPORTS ---
import torch
import gc
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# --- LOAD DATASET ---
dataset = load_dataset("gretelai/synthetic_text_to_sql")
print(f"Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

# --- LOAD MODEL (4-bit quantized) ---
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
print(f"Model loaded: {model.get_memory_footprint()/1e9:.1f} GB")

# --- HELPER FUNCTIONS ---
def create_prompt(sample):
    return f"""[INST] You are a helpful SQL assistant. Given the following database schema and a natural language question, generate the correct SQL query and explain what it does.

### Database Schema:
{sample["sql_context"]}

### Question:
{sample["sql_prompt"]}

### Response:
[/INST]"""

def generate_response(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0], skip_special_tokens=True)
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response

# --- TEST BEFORE FINE-TUNING ---
test_samples = [dataset["test"][i] for i in [0, 10, 50]]

print("\n" + "=" * 60)
print("BEFORE FINE-TUNING")
print("=" * 60)

before_results = []
for i, sample in enumerate(test_samples):
    prompt = create_prompt(sample)
    response = generate_response(prompt)
    print(f"\n--- Example {i+1} ---")
    print(f"Question: {sample['sql_prompt']}")
    print(f"Expected: {sample['sql']}")
    print(f"Model:    {response[:300]}")
    before_results.append({
        "question": sample["sql_prompt"],
        "expected": sample["sql"],
        "output": response,
    })

# --- CLEAR MEMORY BEFORE TRAINING ---
gc.collect()
torch.cuda.empty_cache()

# --- FORMAT TRAINING DATA ---
def format_training_sample(sample):
    return f"""[INST] You are a helpful SQL assistant. Given the following database schema and a natural language question, generate the correct SQL query and explain what it does.

### Database Schema:
{sample["sql_context"]}

### Question:
{sample["sql_prompt"]}

### Response:
[/INST]
**SQL Query:**
{sample["sql"]}

**Explanation:**
{sample["sql_explanation"]}"""

train_data = dataset["train"].shuffle(seed=42).select(range(1000))
train_data = train_data.map(lambda s: {"formatted_text": format_training_sample(s)})
formatted_dataset = train_data.map(
    lambda x: {"text": x["formatted_text"]},
    remove_columns=train_data.column_names,
)
print(f"\nTraining samples: {len(formatted_dataset)}")

# --- CONFIGURE LORA + TRAINING ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

training_args = TrainingArguments(
    output_dir="./mistral-sql-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    optim="paged_adamw_8bit",
    warmup_steps=10,
    max_grad_norm=0.3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")
print("Starting training...\n")

# --- TRAIN ---
trainer.train()
print("\n✅ Training complete!")

# --- TEST AFTER FINE-TUNING ---
model.config.use_cache = True

print("\n" + "=" * 60)
print("AFTER FINE-TUNING")
print("=" * 60)

after_results = []
for i, sample in enumerate(test_samples):
    prompt = create_prompt(sample)
    response = generate_response(prompt)
    print(f"\n--- Example {i+1} ---")
    print(f"Question: {sample['sql_prompt']}")
    print(f"Expected: {sample['sql']}")
    print(f"Model:    {response[:300]}")
    after_results.append({
        "question": sample["sql_prompt"],
        "expected": sample["sql"],
        "output": response,
    })

# --- BEFORE vs AFTER COMPARISON ---
print("\n" + "=" * 60)
print("BEFORE vs AFTER COMPARISON")
print("=" * 60)
for i in range(len(test_samples)):
    print(f"\n{'='*60}")
    print(f"Example {i+1}: {before_results[i]['question']}")
    print(f"{'='*60}")
    print(f"✅ Expected:\n{before_results[i]['expected']}")
    print(f"\n❌ Before fine-tuning:\n{before_results[i]['output'][:300]}")
    print(f"\n🟢 After fine-tuning:\n{after_results[i]['output'][:300]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
GPU: Tesla T4
GPU memory: 15.6 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

synthetic_text_to_sql_train.snappy.parqu(…):   0%|          | 0.00/32.4M [00:00<?, ?B/s]

synthetic_text_to_sql_test.snappy.parque(…):   0%|          | 0.00/1.90M [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Trainable: 13,631,488 / 3,765,702,656 (0.36%)
Starting training...



Step,Training Loss
10,1.166277
20,0.489566
30,0.433016
40,0.427566
50,0.418851
60,0.406885
70,0.405119
80,0.375196


Step,Training Loss
10,1.166277
20,0.489566
30,0.433016
40,0.427566
50,0.418851
60,0.406885
70,0.405119
80,0.375196
90,0.381380
100,0.377157



✅ Training complete!

AFTER FINE-TUNING

--- Example 1 ---
Question: What is the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table?
Expected: SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');
Model:    **SQL Query:**
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

**Explanation:**
This query calculates the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table. It uses the AVG function to c

--- Example 2 ---
Question: How many decentralized applications have been downloaded from the 'Asia-Pacific' region?
Expected: SELECT SUM(dapp_downloads) FROM dapp_ranking WHERE dapp_region = 'Asia-Pacific';
Model:    **SQL Query:**
SELECT COUNT(*) FROM dapp_ranking WHERE dapp_region = 'Asia-Pacific';

**Explanation:**
This query counts the number of decentralized applications down

In [2]:
# --- EVALUATION ON 20 TEST SAMPLES ---
import re

def extract_sql_from_response(response):
    """Extract just the SQL query from model output"""
    # After fine-tuning, format is **SQL Query:** ... **Explanation:**
    if "**SQL Query:**" in response:
        sql = response.split("**SQL Query:**")[1]
        if "**Explanation:**" in sql:
            sql = sql.split("**Explanation:**")[0]
        return sql.strip()
    # Before fine-tuning, SQL might be in ```sql blocks
    if "```sql" in response:
        sql = response.split("```sql")[1]
        if "```" in sql:
            sql = sql.split("```")[0]
        return sql.strip()
    return response.strip()

def simple_sql_match(predicted, expected):
    """Normalize and compare SQL queries"""
    def normalize(s):
        s = s.lower().strip().rstrip(";")
        s = re.sub(r'\s+', ' ', s)
        return s
    return normalize(predicted) == normalize(expected)

# Evaluate on 20 test samples
eval_indices = list(range(0, 200, 10))  # 20 samples spread across test set
correct_before = 0
correct_after = 0
results_table = []

print("Evaluating 20 test samples...")
print("=" * 60)

for idx in eval_indices:
    sample = dataset["test"][idx]
    prompt = create_prompt(sample)

    # Generate after fine-tuning
    after_response = generate_response(prompt)
    after_sql = extract_sql_from_response(after_response)

    # Check match
    expected = sample["sql"]
    match = simple_sql_match(after_sql, expected)
    if match:
        correct_after += 1

    results_table.append({
        "question": sample["sql_prompt"][:80],
        "expected_sql": expected[:80],
        "generated_sql": after_sql[:80],
        "match": "✅" if match else "❌",
    })

print(f"\nAfter fine-tuning accuracy (exact match): {correct_after}/{len(eval_indices)} = {correct_after/len(eval_indices)*100:.0f}%")

# Show results as table
results_df = pd.DataFrame(results_table)
print("\n")
print(results_df.to_string(index=False))

# --- GRADIO INTERFACE ---
def query_model(schema, question):
    sample = {"sql_context": schema, "sql_prompt": question}
    prompt = create_prompt(sample)
    response = generate_response(prompt, max_new_tokens=300)
    return response

demo = gr.Interface(
    fn=query_model,
    inputs=[
        gr.Textbox(
            label="Database Schema",
            placeholder="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
            lines=4,
        ),
        gr.Textbox(
            label="Your Question",
            placeholder="Show me all employees in the sales department",
            lines=2,
        ),
    ],
    outputs=gr.Textbox(label="Generated SQL + Explanation", lines=8),
    title="🔍 Text-to-SQL — Fine-tuned Mistral-7B",
    description="Enter a database schema and ask a question in plain English. The fine-tuned model will generate the SQL query and explain it.",
    examples=[
        [
            "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary REAL);",
            "What is the average salary of employees in each department?"
        ],
        [
            "CREATE TABLE orders (order_id INT, customer_id INT, product TEXT, amount REAL, order_date DATE);",
            "Find the total amount spent by each customer in 2024"
        ],
        [
            "CREATE TABLE students (id INT, name TEXT, grade INT, city TEXT); CREATE TABLE scores (student_id INT, subject TEXT, score INT);",
            "What is the average score of students from New York?"
        ],
    ],
)

print("\nLaunching Gradio interface...")
demo.launch(share=True)

Evaluating 20 test samples...

After fine-tuning accuracy (exact match): 4/20 = 20%


                                                                        question                                                                     expected_sql                                                                    generated_sql match
What is the average explainability score of creative AI applications in 'Europe' SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'No SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'No     ✅
How many decentralized applications have been downloaded from the 'Asia-Pacific' SELECT SUM(dapp_downloads) FROM dapp_ranking WHERE dapp_region = 'Asia-Pacific';            SELECT COUNT(*) FROM dapp_ranking WHERE dapp_region = 'Asia-Pacific';     ❌
            Which rural areas have the highest prevalence of asthma in children? SELECT county, state, AVG(prevalence) AS avg_prevalence FROM asthma WHERE age <  SELEC

NameError: name 'gr' is not defined

In [3]:
import gradio as gr

demo = gr.Interface(
    fn=query_model,
    inputs=[
        gr.Textbox(
            label="Database Schema",
            placeholder="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
            lines=4,
        ),
        gr.Textbox(
            label="Your Question",
            placeholder="Show me all employees in the sales department",
            lines=2,
        ),
    ],
    outputs=gr.Textbox(label="Generated SQL + Explanation", lines=8),
    title="🔍 Text-to-SQL — Fine-tuned Mistral-7B",
    description="Enter a database schema and ask a question in plain English. The fine-tuned model will generate the SQL query and explain it.",
    examples=[
        [
            "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary REAL);",
            "What is the average salary of employees in each department?"
        ],
        [
            "CREATE TABLE orders (order_id INT, customer_id INT, product TEXT, amount REAL, order_date DATE);",
            "Find the total amount spent by each customer in 2024"
        ],
        [
            "CREATE TABLE students (id INT, name TEXT, grade INT, city TEXT); CREATE TABLE scores (student_id INT, subject TEXT, score INT);",
            "What is the average score of students from New York?"
        ],
    ],
)

print("Launching Gradio interface...")
demo.launch(share=True)

Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c023a11fb587761c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
